In [56]:
import pickle
import numpy as np
import pandas as pd
import sklearn as sk
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.metrics import mean_squared_error

In [10]:
URL1 = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2021-01.parquet"
URL2 = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2021-02.parquet"

In [38]:
def read_dataframe(url):
    df = pd.read_parquet(url)
    df.tpep_dropoff_datetime = pd.to_datetime(df.tpep_dropoff_datetime)
    df.tpep_pickup_datetime = pd.to_datetime(df.tpep_pickup_datetime)
    df['duration'] = (df.tpep_dropoff_datetime - df.tpep_pickup_datetime).dt.total_seconds() / 60
    df = df.loc[(df['duration'] > 1) & (df['duration'] <= 60)]
    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)
    return df.iloc[:500000]
    

In [49]:
df_train = read_dataframe(URL1)
df_val = read_dataframe(URL2)
len(df_train), len(df_val)

(500000, 500000)

In [50]:
df_train['PU_DO'] = df_train['PULocationID'] + '_' +  df_train['DOLocationID']
df_val['PU_DO'] = df_val['PULocationID'] + '_' +  df_val['DOLocationID']

AttributeError: 'DataFrame' object has no attribute 'indices'

In [59]:
numerical = ['trip_distance']
categorical = ['PU_DO']

dv = DictVectorizer()

train_dicts = df_train[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dicts)

X_train.indices = X_train.indices.astype(np.int32).astype(np.int32)
X_train.indptr = X_train.indptr.astype(np.int32).astype(np.int32)

val_dicts = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dicts)
X_val, X_train

target = 'duration'
y_train = df_train[target]


In [60]:
lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred = lr.predict(X_val)

mean_squared_error(y_val, y_pred)

25.70617212516012

In [61]:
with open('models/lin_reg.bin', 'wb') as f_out:
    pickle.dump((dv, lr), f_out)

In [63]:
lr = Lasso(0.1)
lr.fit(X_train, y_train)

y_pred = lr.predict(X_val)

mean_squared_error(y_val, y_pred)

36.67658380324457